# 03 — Recorded impacts (EM-DAT) and storm cross-reference

What the events cost — people affected, deaths, damage — by hazard type and over
time, then joined back to the storms from `01_windspeed_ibtracs.ipynb`.

**Run third.** Reads the EM-DAT export and `outputs/ibtracs_impact_storms_*.csv`.

> **Reporting bias.** EM-DAT completeness improves sharply after ~1980; decade
> comparisons partly measure reporting effort. Read the early record as a floor.

> **Licence.** EM-DAT may not be redistributed — keep the `.xlsx` out of the
> repository. Only derived aggregates are shareable, with citation.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Repo root on sys.path, whether launched from notebooks/ or the repo root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker

import config
from plot_utils import FALLBACK_COLOR, normalise_cat, save_fig, set_style

config.ensure_dirs()
set_style(context="notebook", font_scale=1.0)

# EM-DAT column names, in one place — long and easy to mistype
DAMAGE_COL   = "Total Damage, Adjusted ('000 US$)"
AFFECTED_COL = "Total Affected"
DEATHS_COL   = "Total Deaths"

HAZARD_COLORS = {
    "Storm": "#3B0086",
    "Flood": "#5B9BD5",
    "Volcanic activity": "#C1440E",
    "Other": "#B4B2A9",
}
MAIN_TYPES = ["Storm", "Flood", "Volcanic activity"]
ISO3 = config.ISO3

print(config.summary())
print(f"\nEM-DAT file : {config.EMDAT_XLSX}")

### Input data

EM-DAT needs a free account and a custom country request
(<https://public.emdat.be/data>). Save the export to `config.EMDAT_XLSX`.

In [ ]:
data = pd.read_excel(config.EMDAT_XLSX)

for col in [AFFECTED_COL, DEATHS_COL, DAMAGE_COL]:
    data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0)
data["Disaster Type"] = data["Disaster Type"].fillna("Unknown")

# Filter first, so everything downstream describes the same event set
data = data[~data["Disaster Type"].isin(config.EMDAT_EXCLUDE_TYPES)].copy()

first_year = int(data.loc[data[AFFECTED_COL] > 0, "Start Year"].min())
last_year  = int(data["Start Year"].max())

print(f"[OK] {len(data)} events for {config.COUNTRY_NAME}")
print(f"     Hazard types : {sorted(data['Disaster Type'].unique())}")
print(f"     First year with affected-population data : {first_year}")

In [ ]:
print("Events ranked by people affected:")
print(data.sort_values(AFFECTED_COL, ascending=False)
      .loc[:, ["Event Name", "Disaster Type", "Start Year", AFFECTED_COL, DEATHS_COL]]
      .head(15).to_string(index=False))

## 2. Aggregate by hazard type

In [ ]:
by_type = (
    data.groupby("Disaster Type")
    .agg(
        event_count=("DisNo.", "count"),
        total_affected=(AFFECTED_COL, "sum"),
        total_deaths=(DEATHS_COL, "sum"),
        total_damage=(DAMAGE_COL, "sum"),
    )
    .sort_values("total_affected", ascending=False)
    .reset_index()
)
for col in ["total_affected", "total_deaths", "total_damage"]:
    by_type[col] = by_type[col].astype(int)
by_type["pct_affected"] = (
    by_type["total_affected"] / by_type["total_affected"].sum() * 100
).round(1)

print(f"Period          : {first_year} – {last_year}")
print(f"Total events    : {data['DisNo.'].nunique():,}")
print(f"Total affected  : {int(data[AFFECTED_COL].sum()):,}")
print(f"Total deaths    : {int(data[DEATHS_COL].sum()):,}")
print(f"Total damage    : ${data[DAMAGE_COL].sum() / 1_000:,.0f}M (adjusted USD)")
print("\nBy hazard type:")
print(by_type.to_string(index=False))

summary_path = config.OUTPUT_DIR / f"summary_by_hazard_{ISO3}.csv"
by_type.to_csv(summary_path, index=False)
print(f"\n  Saved → {summary_path}")

In [ ]:
# Data-quality audit: anything with a low "% with data" is not a quotable total
audit = [
    {
        "Metric": label,
        "Total events": len(data),
        "Zero or missing": int((data[col] == 0).sum()),
        "Has data": len(data) - int((data[col] == 0).sum()),
        "% with data": round((len(data) - int((data[col] == 0).sum())) / len(data) * 100, 1),
    }
    for col, label in [(AFFECTED_COL, "Affected"), (DEATHS_COL, "Deaths"),
                       (DAMAGE_COL, "Damage")]
]
print(pd.DataFrame(audit).to_string(index=False))

## 3. Impact by hazard type

In [ ]:
def hazard_bar(metric, title, ylabel, filename):
    """Bar chart of one impact metric by hazard type."""
    plot_df = (by_type.loc[by_type[metric] > 0, ["Disaster Type", metric]]
               .sort_values(metric, ascending=False))
    colors = plot_df["Disaster Type"].map(HAZARD_COLORS).fillna(FALLBACK_COLOR).tolist()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.set_axisbelow(True)
    bars = ax.bar(plot_df["Disaster Type"], plot_df[metric],
                  color=colors, width=0.7, alpha=0.88)
    for bar, color in zip(bars, colors):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                f"{bar.get_height():,.0f}", ha="center", va="bottom",
                color=color, fontsize=9, fontweight="bold")

    ax.set_title(title, fontsize=11, color="#3d3d3a", pad=10)
    ax.set_ylabel(ylabel, fontsize=10, color="#3d3d3a", labelpad=8)
    ax.tick_params(colors="#888780", labelsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.5, color="#d3d1c7")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#d3d1c7")
    plt.tight_layout()
    save_fig(fig, filename, config.OUTPUT_DIR)
    plt.show()


hazard_bar("total_affected", "People affected by hazard type", "Total affected",
           f"3a_affected_by_hazard_{ISO3}.png")
hazard_bar("total_deaths", "Deaths by hazard type", "Total deaths",
           f"3b_deaths_by_hazard_{ISO3}.png")
hazard_bar("total_damage", "Total adjusted damage by hazard type", "'000 USD",
           f"3c_damage_by_hazard_{ISO3}.png")

## 4. Impact over time

In [ ]:
def prepare_timeline(source):
    """Events from `first_year` on, with a plotting colour per hazard type."""
    out = source[source["Start Year"] >= first_year].copy()
    out["hazard_group"] = out["Disaster Type"].where(
        out["Disaster Type"].isin(MAIN_TYPES), "Other")
    out["color"] = out["hazard_group"].map(HAZARD_COLORS).fillna(FALLBACK_COLOR)
    return out


def timeline_chart(plot_df, filename, title, badge_col=None):
    """One bar per event by year, labelled with the event name; `badge_col`
    adds the matched Saffir-Simpson category above the bar."""
    fig, ax = plt.subplots(figsize=(16, 6))
    ax.set_axisbelow(True)

    y_max = plot_df[AFFECTED_COL].max()
    ax.set_ylim(0, y_max * (1.25 if badge_col else 1.15))
    bar_width = 0.8

    for _, row in plot_df.iterrows():
        val, yr, color = row[AFFECTED_COL], row["Start Year"], row["color"]
        ax.bar(yr, val, width=bar_width, color=color, alpha=0.88, zorder=2, linewidth=0)

        name = row["Event Name"]
        label = name if pd.notna(name) else row["Disaster Type"]
        ax.text(yr - bar_width / 2, val + y_max * 0.01, label,
                ha="left", va="bottom", fontsize=7.5, rotation=45, color=color)

        if badge_col and pd.notna(row.get(badge_col)) and val > 0:
            ax.text(yr, val + y_max * 0.06, normalise_cat(row[badge_col]),
                    ha="center", va="bottom", fontsize=7, color="#3d3d3a",
                    bbox=dict(boxstyle="round,pad=0.2", facecolor="white",
                              edgecolor=color, linewidth=0.8))

    handles = [mpatches.Patch(facecolor=v, label=k, linewidth=0)
               for k, v in HAZARD_COLORS.items() if k in set(plot_df["hazard_group"])]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.1),
              ncol=4, frameon=False, labelcolor="#3d3d3a",
              handlelength=1.5, handleheight=1.5, borderpad=0.8)

    ax.set_xlim(first_year - 2, plot_df["Start Year"].max() + 1)
    ax.set_xlabel("Year", fontsize=10, color="#3d3d3a", labelpad=8)
    ax.set_ylabel("Total people affected", fontsize=10, color="#3d3d3a", labelpad=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax.tick_params(colors="#888780", labelsize=9)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.5, color="#d3d1c7", zorder=1)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#d3d1c7")
    ax.set_title(title, fontsize=11, color="#3d3d3a", pad=12)

    plt.tight_layout()
    save_fig(fig, filename, config.OUTPUT_DIR)
    plt.show()


timeline_chart(prepare_timeline(data), f"4a_affected_by_year_{ISO3}.png",
               f"People affected by disaster event — {config.COUNTRY_NAME}")

In [ ]:
by_decade = (
    data.assign(decade=(data["Start Year"] // 10 * 10).astype(int))
    .groupby("decade")
    .agg(
        event_count=("DisNo.", "count"),
        total_affected=(AFFECTED_COL, "sum"),
        total_deaths=(DEATHS_COL, "sum"),
        total_damage=(DAMAGE_COL, "sum"),
    )
    .reset_index()
)
print(by_decade.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (col, label, color) in zip(axes, [
    ("total_affected", "People affected", "#3B0086"),
    ("total_deaths", "Deaths", "#C1440E"),
    ("total_damage", "Damage ('000 USD)", "#5B9BD5"),
]):
    ax.set_axisbelow(True)
    bars = ax.bar(by_decade["decade"].astype(str), by_decade[col],
                  color=color, alpha=0.88, width=0.6)
    for bar in bars:
        if bar.get_height() > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                    f"{bar.get_height():,.0f}", ha="center", va="bottom",
                    fontsize=8, color=color)
    ax.set_title(label, fontsize=10, color="#3d3d3a", pad=8)
    ax.set_xlabel("Decade", fontsize=9, color="#3d3d3a")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
    ax.tick_params(colors="#888780", labelsize=8)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.5, color="#d3d1c7")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#d3d1c7")

plt.suptitle(f"Disaster impacts by decade — {config.COUNTRY_NAME}  "
             "(early decades under-reported)", fontsize=11, color="#3d3d3a", y=1.02)
plt.tight_layout()
save_fig(fig, f"4b_decade_aggregation_{ISO3}.png", config.OUTPUT_DIR)
plt.show()

## 5. Empirical return periods by impact

The *n*th worst event on record has been equalled or exceeded *n* times in the
record length, so its return period is roughly `record_years / n`. Empirical, no
distribution fitted, and the early record is incomplete.

In [ ]:
record_years = last_year - first_year + 1

rp = (data[data[AFFECTED_COL] > 0]
      .sort_values(AFFECTED_COL, ascending=False)
      .reset_index(drop=True))
rp["rank"] = rp.index + 1
rp["return_period_yr"] = (record_years / rp["rank"]).round(1)
rp["label"] = rp["Event Name"].fillna(rp["Disaster Type"])

print(f"Record length: {record_years} years ({first_year}–{last_year})\n")
print(rp[["label", "Start Year", AFFECTED_COL, "return_period_yr"]].to_string(index=False))

## 6. Cross-reference with the storm record

Matching is on **date proximity**, not year: EM-DAT start dates are compared to
each storm's closest approach within `config.EMDAT_MATCH_TOLERANCE_DAYS`. Year
alone is ambiguous in seasons with more than one qualifying storm. Where EM-DAT
gives no month or day, the match falls back to year and is flagged — check those
by hand before quoting them.

In [ ]:
storms = pd.read_csv(config.STORMS_CSV, parse_dates=["closest_approach_date"])
storms["ss_category"] = storms["ss_category"].apply(normalise_cat)
print(f"[OK] {len(storms)} storms from {config.STORMS_CSV.name}")
print(storms[["name", "year", "max_wind_kmh", "ss_category", "closest_dist_km"]]
      .to_string(index=False))

emdat_storms = data[data["Disaster Type"] == "Storm"].copy()
if emdat_storms.empty:
    raise ValueError("No EM-DAT entries of type 'Storm' — nothing to cross-reference.")

month = pd.to_numeric(emdat_storms.get("Start Month"), errors="coerce")
day = pd.to_numeric(emdat_storms.get("Start Day"), errors="coerce")
emdat_storms["has_full_date"] = month.notna() & day.notna()
emdat_storms["start_date"] = pd.to_datetime(
    dict(year=emdat_storms["Start Year"], month=month.fillna(1), day=day.fillna(1)),
    errors="coerce",
)
print(f"\n{len(emdat_storms)} EM-DAT storm entries "
      f"({int(emdat_storms['has_full_date'].sum())} with a full start date)")

In [ ]:
def match_storm(row):
    """Nearest closest-approach date within tolerance, else a same-year match
    flagged as ambiguous."""
    same_year = storms[storms["year"] == row["Start Year"]]
    if same_year.empty:
        return pd.Series({"match_name": np.nan, "match_category": np.nan,
                          "match_wind_kmh": np.nan, "match_dist_km": np.nan,
                          "match_quality": "no match"})

    if row["has_full_date"] and pd.notna(row["start_date"]):
        delta = (same_year["closest_approach_date"] - row["start_date"]).abs()
        best = same_year.loc[delta.idxmin()]
        quality = (f"date (±{delta.min().days}d)"
                   if delta.min() <= pd.Timedelta(days=config.EMDAT_MATCH_TOLERANCE_DAYS)
                   else f"year only (nearest storm {delta.min().days}d away)")
    else:
        best = same_year.iloc[0]
        quality = "year only" + (" — ambiguous" if len(same_year) > 1 else "")

    return pd.Series({
        "match_name": best["name"],
        "match_category": best["ss_category"],
        "match_wind_kmh": best["max_wind_kmh"],
        "match_dist_km": best["closest_dist_km"],
        "match_quality": quality,
    })


matched = emdat_storms.join(emdat_storms.apply(match_storm, axis=1))

print("EM-DAT storms matched to the IBTrACS impact set:")
print(matched[["Event Name", "Start Year", AFFECTED_COL, DEATHS_COL,
               "match_name", "match_category", "match_wind_kmh", "match_quality"]]
      .to_string(index=False))

unmatched = int((matched["match_quality"] == "no match").sum())
if unmatched:
    print(f"\n[note] {unmatched} EM-DAT storm(s) with no storm in the impact set that "
          f"year — below the intensity threshold, or outside the "
          f"{config.IMPACT_RADIUS_KM:.0f} km buffer.")

In [ ]:
# Rebuilt from `data` rather than re-merging, so re-running cannot create
# duplicate _x / _y columns.
timeline_matched = prepare_timeline(data).merge(
    matched[["DisNo.", "match_category", "match_name"]], on="DisNo.", how="left")

timeline_chart(timeline_matched, f"4c_affected_by_year_with_category_{ISO3}.png",
               f"People affected, with matched storm category — {config.COUNTRY_NAME}",
               badge_col="match_category")

---

## Reading the three together

* `01` — how often a storm of a given category comes close.
* `02` — how much rain those events delivered, and the exceedance curve a
  threshold can be anchored to.
* `03` — what was recorded as impact, i.e. whether the hazard set picks up the
  events that actually hurt.

An EM-DAT event with no storm match, or a matched storm with no EM-DAT entry,
is itself informative: either the threshold is set too high, or the impact went
unrecorded.